# Phase 1 - Bank Marketing Dataset Exploration

**Project:** Bank Marketing Intelligence System (ISM)
**Scope:** Dataset understanding, data quality, descriptive statistics, customer analysis, and EDA visualisations. No machine-learning models are trained.

The reusable implementation lives in `src/phase1_exploration.py`; this notebook imports its functions so that the analysis and the script stay consistent.

## Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
# Allow the notebook to be launched from the project root or the notebooks folder.
while not (PROJECT_ROOT / "src" / "phase1_exploration.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not locate the project root (src/phase1_exploration.py).")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import phase1_exploration as p1

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "phase1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset     :", DATA_PATH)

## Task A - Dataset loading

In [ ]:
df = p1.load_dataset(DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())
display(df.tail())
display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).to_numpy(),
    "non_null": df.notna().sum().to_numpy(),
    "unique": df.nunique(dropna=True).to_numpy(),
}))

## Task B - Data quality

In [ ]:
quality = p1.analyse_quality(df)
display(quality["missing"])
display(quality["unknown"])
print("Duplicate rows      :", quality["duplicate_rows"])
print("pdays = -1          :", quality["pdays_minus_one"])
print("pdays/previous flags:", quality["pdays_previous_inconsistent"])

## Task C - Statistical analysis

In [ ]:
statistics = p1.analyse_statistics(df)
display(statistics["numeric"].round(2))
display(statistics["target"])
display(statistics["categorical"].groupby("column").head(5))

## Task D - Customer analysis

In [ ]:
enriched = p1.add_customer_segments(df)
tables = p1.build_rate_tables(enriched)
p1.print_customer_analysis(enriched, tables)

print("Age group rates:")
display(tables["age_group"].round(2))
print("Occupation rates:")
display(tables["job"].round(2))
print("Previous campaign outcome rates:")
display(tables["poutcome"].round(2))

## Task E - Visualisations

In [ ]:
from IPython.display import Image, display as display_image

p1.set_plot_style(dpi=150)
chart_paths = p1.create_visualizations(enriched, statistics, tables, OUTPUT_DIR)

for chart in chart_paths:
    print(chart.name)
    display_image(Image(filename=str(chart), width=900))

## Task F - Important data-mining considerations

- **Class imbalance:** 11.70% of contacts subscribed (7.55 non-subscribers per subscriber). A constant "no" model already scores 88.30% accuracy while producing no leads.
- **Accuracy is misleading:** the business cares about finding likely subscribers (recall, precision, lift) and about campaign cost, not raw accuracy.
- **'unknown' is information:** 81.75% of `poutcome` and 28.80% of `contact` values are unknown. Deleting them would remove large, non-random groups; keep them as explicit categories and investigate.
- **pdays = -1:** sentinel for "never contacted before" (81.74% of records). It must not be averaged as an elapsed-day value; use a contact-history flag.
- **duration leakage:** call duration is known only after the call and correlates +0.395 with the target. Exclude it from pre-call targeting models.

## Reproduce the full pipeline

Run the script from the project root to regenerate every CSV, chart, and the Markdown report:

```bash
python src/phase1_exploration.py
```